## Ecommerce Pipeline – Pytest Test Suite

This notebook validates the **entire ecommerce data-analysis pipeline** using
**pytest**.  Each section defines standard `test_*` functions that are collected
automatically and executed via `pytest.main()`.

**Test Categories**
| # | Category | What is tested |
|---|----------|----------------|
| 1 | Readers | csv_reader, json_reader – happy path + error paths |
| 2 | Schema Helpers | apply_schema – rename, missing key, partial mapping |
| 3 | Delta Writer | delta_writer – write, overwrite, None DataFrame |
| 4 | Data Quality | validate_not_null, no_duplicates, schema, row_count |
| 5 | clean_dataframe | Null filtering, fill defaults, dedup, cast, date parse |
| 6 | Ingestion | ingest_data, run_ingestion – end-to-end + bad format |
| 7 | Transformation | Dim/Fact cleaning logic on realistic data |
| 8 | Enrichment | Join correctness, duplicate handling |

> ⚠️ All test tables use a `_test_` prefix in **az_ci_de_dbs.ecom_dev** and are
> cleaned up at the end.

In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_analysis/resources/utils

In [0]:
"""
Pytest-based test suite for the ecommerce data pipeline.

Test functions are defined as standard `test_*` functions in subsequent cells.
The 'Execute all tests' cell dynamically collects them, generates a
self-contained test module, and runs pytest programmatically.
"""

import pytest
import inspect
import os
import json
import textwrap

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, DecimalType, LongType,
)

# ---------------------------------------------------------------------------
# Test helpers
# ---------------------------------------------------------------------------
TEST_CATALOG = "az_ci_de_dbs"
TEST_SCHEMA  = "ecom_dev"

def _tbl(name: str) -> str:
    """Generate a fully-qualified test table name with _test_ prefix."""
    return f"{TEST_CATALOG}.{TEST_SCHEMA}._test_{name}"

In [0]:
# -----------------------------------------------------------------------
# 1. Reader Tests
# -----------------------------------------------------------------------

def test_csv_reader_happy_path():
    """csv_reader returns a DataFrame with correct columns from Products.csv."""
    df = csv_reader(spark, "/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Products.csv")
    assert isinstance(df, DataFrame), "Expected a DataFrame"
    assert df.count() > 0, "DataFrame is empty"
    assert "Product ID" in df.columns, f"Missing 'Product ID' column; got {df.columns}"

def test_json_reader_happy_path():
    """json_reader returns a non-empty DataFrame from Orders.json."""
    df = json_reader(spark, "/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Orders.json", multi_line=True)
    assert isinstance(df, DataFrame), "Expected a DataFrame"
    assert df.count() > 0, "DataFrame is empty"

def test_csv_reader_file_not_found():
    """csv_reader raises FileNotFoundError or RuntimeError for a non-existent path."""
    with pytest.raises((FileNotFoundError, RuntimeError)):
        csv_reader(spark, "/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/NO_SUCH_FILE.csv")

def test_json_reader_file_not_found():
    """json_reader raises FileNotFoundError or RuntimeError for a non-existent path."""
    with pytest.raises((FileNotFoundError, RuntimeError)):
        json_reader(spark, "/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/NO_SUCH.json")

In [0]:
# -----------------------------------------------------------------------
# 2. Schema Helper Tests
# -----------------------------------------------------------------------

def test_apply_schema_renames_columns():
    """apply_schema correctly renames columns based on mapping."""
    df = spark.createDataFrame([(1, "A")], ["Product ID", "Category"])
    mapping = {"Products": {"Product ID": "product_id", "Category": "category"}}
    result = apply_schema(df, "Products", mapping)
    assert result.columns == ["product_id", "category"], f"Got {result.columns}"

def test_apply_schema_partial_mapping():
    """Columns not in the mapping keep their original names."""
    df = spark.createDataFrame([(1, "A", "X")], ["id", "name", "extra"])
    mapping = {"t": {"id": "ID", "name": "NAME"}}
    result = apply_schema(df, "t", mapping)
    assert result.columns == ["ID", "NAME", "extra"]

def test_apply_schema_missing_key_raises():
    """apply_schema raises KeyError for an unknown alias key."""
    df = spark.createDataFrame([(1,)], ["id"])
    with pytest.raises(KeyError):
        apply_schema(df, "DOES_NOT_EXIST", {"other": {}})

In [0]:
# -----------------------------------------------------------------------
# 3. Delta Writer Tests
# -----------------------------------------------------------------------

def test_delta_writer_creates_table():
    """delta_writer creates a readable Delta table."""
    df = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "val"])
    tbl = _tbl("writer_basic")
    delta_writer(df, tbl)
    result = spark.read.table(tbl)
    assert result.count() == 2

def test_delta_writer_overwrites_existing():
    """delta_writer in overwrite mode replaces all data."""
    tbl = _tbl("writer_overwrite")
    df1 = spark.createDataFrame([(1,)], ["id"])
    delta_writer(df1, tbl)
    df2 = spark.createDataFrame([(10,), (20,)], ["id"])
    delta_writer(df2, tbl)
    assert spark.read.table(tbl).count() == 2, "Table not overwritten"

def test_delta_writer_none_df_raises():
    """delta_writer raises ValueError when df is None."""
    with pytest.raises(ValueError):
        delta_writer(None, _tbl("writer_none"))

def test_delta_writer_empty_table_name_raises():
    """delta_writer raises ValueError when table_name is empty."""
    df = spark.createDataFrame([(1,)], ["id"])
    with pytest.raises(ValueError):
        delta_writer(df, "")

In [0]:
# -----------------------------------------------------------------------
# 4. Data Quality Validation Tests
# -----------------------------------------------------------------------

def test_validate_not_null_passes():
    """validate_not_null passes when no NULLs are present."""
    df = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "name"])
    result = validate_not_null(df, ["id", "name"], label="test")
    assert result == {}

def test_validate_not_null_detects_nulls():
    """validate_not_null raises DataQualityError on NULLs."""
    df = spark.createDataFrame([(1, None), (2, "b")], ["id", "name"])
    with pytest.raises(DataQualityError, match="name"):
        validate_not_null(df, ["name"], label="test")

def test_validate_not_null_missing_column():
    """validate_not_null raises DataQualityError for non-existent column."""
    df = spark.createDataFrame([(1,)], ["id"])
    with pytest.raises(DataQualityError):
        validate_not_null(df, ["missing_col"], label="test")

def test_validate_no_duplicates_passes():
    """validate_no_duplicates passes for unique keys."""
    df = spark.createDataFrame([(1,), (2,), (3,)], ["id"])
    assert validate_no_duplicates(df, ["id"], label="test") == 0

def test_validate_no_duplicates_detects_dupes():
    """validate_no_duplicates raises on duplicate keys."""
    df = spark.createDataFrame([(1,), (1,), (2,)], ["id"])
    with pytest.raises(DataQualityError, match="(?i)duplicate"):
        validate_no_duplicates(df, ["id"], label="test")

def test_validate_schema_passes():
    """validate_schema passes when types match."""
    df = spark.createDataFrame([(1, "a")], ["id", "name"])
    expected = StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
    ])
    assert validate_schema(df, expected, label="test") is True

def test_validate_schema_detects_type_mismatch():
    """validate_schema raises on type mismatch."""
    df = spark.createDataFrame([(1, "a")], ["id", "name"])
    wrong = StructType([
        StructField("id", StringType()),  # actually LongType
    ])
    with pytest.raises(DataQualityError, match="(?i)mismatch"):
        validate_schema(df, wrong, label="test")

def test_validate_schema_detects_missing_column():
    """validate_schema raises when an expected column is absent."""
    df = spark.createDataFrame([(1,)], ["id"])
    expected = StructType([StructField("missing", StringType())])
    with pytest.raises(DataQualityError):
        validate_schema(df, expected, label="test")

def test_validate_row_count_passes():
    """validate_row_count passes for count within range."""
    df = spark.createDataFrame([(i,) for i in range(5)], ["id"])
    assert validate_row_count(df, min_rows=1, max_rows=10, label="test") == 5

def test_validate_row_count_too_few():
    """validate_row_count raises when count < min_rows."""
    df = spark.createDataFrame([], StructType([StructField("id", IntegerType())]))
    with pytest.raises(DataQualityError):
        validate_row_count(df, min_rows=1, label="test")

def test_run_quality_checks_mixed():
    """run_quality_checks returns PASSED and failure messages."""
    df = spark.createDataFrame([(1, None)], ["id", "name"])
    checks = [
        {"check": "row_count", "min_rows": 1},
        {"check": "not_null", "columns": ["name"]},
    ]
    results = run_quality_checks(df, checks, label="mixed")
    assert results["row_count"] == "PASSED"
    assert results["not_null"] != "PASSED"

In [0]:
# -----------------------------------------------------------------------
# 5. clean_dataframe Tests
# -----------------------------------------------------------------------

def test_clean_dataframe_filters_nulls():
    """clean_dataframe drops rows where primary key is NULL."""
    df = spark.createDataFrame([(1, "a"), (None, "b"), (3, "c")], ["id", "name"])
    result = clean_dataframe(df, primary_key="id")
    assert result.count() == 2

def test_clean_dataframe_fills_defaults():
    """clean_dataframe fills NULL values with provided defaults."""
    df = spark.createDataFrame([(1, None)], ["id", "name"])
    result = clean_dataframe(df, primary_key="id", fill_defaults={"name": "unknown"})
    assert result.collect()[0]["name"] == "unknown"

def test_clean_dataframe_deduplicates():
    """clean_dataframe removes exact duplicate rows."""
    df = spark.createDataFrame([(1, "a"), (1, "a"), (2, "b")], ["id", "name"])
    result = clean_dataframe(df, primary_key="id")
    assert result.count() == 2

def test_clean_dataframe_casts_schema():
    """clean_dataframe casts columns to the target schema."""
    df = spark.createDataFrame([("1", "2.5")], ["id", "price"])
    schema = StructType([
        StructField("id", IntegerType()),
        StructField("price", DoubleType()),
    ])
    result = clean_dataframe(df, primary_key="id", target_schema=schema)
    row = result.collect()[0]
    assert isinstance(row["id"], int), f"Expected int, got {type(row['id'])}"
    assert isinstance(row["price"], float), f"Expected float, got {type(row['price'])}"

def test_clean_dataframe_parses_dates():
    """clean_dataframe converts date strings using provided format."""
    from datetime import date
    df = spark.createDataFrame([("1", "15/3/2024")], ["id", "order_date"])
    schema = StructType([
        StructField("id", StringType()),
        StructField("order_date", DateType()),
    ])
    result = clean_dataframe(
        df, primary_key="id",
        target_schema=schema,
        date_columns={"order_date": "d/M/yyyy"}
    )
    row = result.collect()[0]
    assert row["order_date"] == date(2024, 3, 15)

def test_clean_dataframe_invalid_pk_raises():
    """clean_dataframe raises ValueError for a non-existent primary key."""
    df = spark.createDataFrame([(1,)], ["id"])
    with pytest.raises(ValueError):
        clean_dataframe(df, primary_key="missing_pk")

def test_clean_dataframe_all_nulls_on_pk():
    """clean_dataframe returns empty DF when all PKs are NULL."""
    df = spark.createDataFrame([(None, "a"), (None, "b")], ["id", "name"])
    result = clean_dataframe(df, primary_key="id")
    assert result.count() == 0

In [0]:
# -----------------------------------------------------------------------
# 6. Ingestion Tests
# -----------------------------------------------------------------------

def test_ingest_data_csv_end_to_end():
    """ingest_data reads CSV, applies aliases, and writes a Delta table."""
    alias_dict = {"Products": {"Product ID": "product_id", "Category": "category",
                                "Sub-Category": "sub_category", "Product Name": "product_name",
                                "State": "state", "Price per product": "price_per_product"}}
    tbl = _tbl("ingest_csv")
    df = ingest_data(
        spark,
        file_path="/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Products.csv",
        file_format="csv",
        target_table=tbl,
        alias_key="Products",
        column_alias_dict=alias_dict,
        reader_options={"header": True},
    )
    assert "product_id" in df.columns, f"Alias not applied: {df.columns}"
    persisted = spark.read.table(tbl)
    assert persisted.count() == df.count()

def test_ingest_data_unsupported_format():
    """ingest_data raises ValueError for an unsupported format."""
    with pytest.raises(ValueError, match="Unsupported"):
        ingest_data(spark, "/dummy", "parquet_v99", _tbl("bad_fmt"))

def test_run_ingestion_empty_config():
    """run_ingestion returns empty dict for empty config list."""
    result = run_ingestion(spark, [])
    assert result == {}

def test_run_ingestion_missing_required_key():
    """run_ingestion raises ValueError for config missing required keys."""
    bad_config = [{"file_path": "/dummy"}]  # missing file_format, target_table
    with pytest.raises(ValueError, match="(?i)missing required keys"):
        run_ingestion(spark, bad_config)

In [0]:
# -----------------------------------------------------------------------
# 7. Transformation Logic Tests
# -----------------------------------------------------------------------

def test_transform_products_removes_null_pk():
    """Product transformation filters out rows with NULL product_id."""
    data = [("P1", "Cat", "Sub", "Name", "CA", 10.0),
            (None, "Cat", "Sub", "Name", "CA", 5.0)]
    df = spark.createDataFrame(data, ["product_id", "category", "sub_category",
                                       "product_name", "state", "price_per_product"])
    result = clean_dataframe(df, primary_key="product_id",
                             fill_defaults={"category": "unknown", "price_per_product": 0})
    assert result.count() == 1
    assert result.collect()[0]["product_id"] == "P1"

def test_transform_orders_date_parsing():
    """Order transformation correctly parses d/M/yyyy dates."""
    from datetime import date
    data = [("O1", "5/1/2023", "10/1/2023", "Standard", "C1", "P1", 2, 100.0, 0.1, 50.0)]
    cols = ["order_id", "order_date", "ship_date", "ship_mode",
            "customer_id", "product_id", "quantity", "price", "discount", "profit"]
    df = spark.createDataFrame(data, cols)
    schema = StructType([
        StructField("order_id", StringType()),
        StructField("order_date", DateType()),
        StructField("ship_date", DateType()),
        StructField("ship_mode", StringType()),
        StructField("customer_id", StringType()),
        StructField("product_id", StringType()),
        StructField("quantity", IntegerType()),
        StructField("price", DoubleType()),
        StructField("discount", DoubleType()),
        StructField("profit", DecimalType(10, 2)),
    ])
    result = clean_dataframe(
        df, primary_key="order_id",
        target_schema=schema,
        date_columns={"order_date": "d/M/yyyy", "ship_date": "d/M/yyyy"},
    )
    row = result.collect()[0]
    assert row["order_date"] == date(2023, 1, 5)
    assert row["ship_date"] == date(2023, 1, 10)

def test_transform_customers_fills_defaults():
    """Customer transformation fills NULL fields with 'unknown'."""
    data = [("C1", None, None, None, None, None, None, None, None, None, None)]
    cols = ["customer_id", "customer_name", "email", "phone", "address",
            "segment", "country", "city", "state", "postal_code", "region"]
    df = spark.createDataFrame(data, cols)
    defaults = {c: "unknown" for c in cols if c != "customer_id"}
    result = clean_dataframe(df, primary_key="customer_id", fill_defaults=defaults)
    row = result.collect()[0]
    assert row["customer_name"] == "unknown"
    assert row["email"] == "unknown"
    assert row["region"] == "unknown"

In [0]:
# -----------------------------------------------------------------------
# 8. Enrichment Logic Tests
# -----------------------------------------------------------------------

def test_enrich_join_produces_correct_columns():
    """Enrichment join produces expected output columns."""
    orders = spark.createDataFrame(
        [("O1", "C1", "P1", 100.0)],
        ["order_id", "customer_id", "product_id", "profit"],
    )
    customers = spark.createDataFrame(
        [("C1", "Alice", "US")],
        ["customer_id", "customer_name", "country"],
    )
    products = spark.createDataFrame(
        [("P1", "Tech", "Phones")],
        ["product_id", "category", "sub_category"],
    )
    enriched = (
        orders.alias("o")
        .join(customers.alias("c"), on="customer_id", how="inner")
        .join(products.alias("p"), on="product_id", how="inner")
        .select("o.*", "c.customer_name", "c.country", "p.category", "p.sub_category")
    )
    assert enriched.count() == 1
    assert "customer_name" in enriched.columns
    assert "category" in enriched.columns
    row = enriched.collect()[0]
    assert row["customer_name"] == "Alice"

def test_enrich_join_drops_unmatched():
    """Inner join correctly excludes orders with no matching dimension."""
    orders = spark.createDataFrame(
        [("O1", "C1", "P1", 100.0), ("O2", "C_MISSING", "P1", 50.0)],
        ["order_id", "customer_id", "product_id", "profit"],
    )
    customers = spark.createDataFrame([("C1", "Alice", "US")], ["customer_id", "customer_name", "country"])
    products  = spark.createDataFrame([("P1", "Tech", "Phones")], ["product_id", "category", "sub_category"])
    enriched = (
        orders.alias("o")
        .join(customers.alias("c"), on="customer_id", how="inner")
        .join(products.alias("p"), on="product_id", how="inner")
        .select("o.*", "c.customer_name", "c.country", "p.category", "p.sub_category")
    )
    assert enriched.count() == 1, f"Expected 1 row after inner join, got {enriched.count()}"

def test_profit_aggregation():
    """Profit aggregation groups correctly by year, category, customer."""
    from datetime import date
    data = [
        ("O1", date(2023, 1, 1), "C1", "P1", "Alice", "Tech", "Phones", 100.0),
        ("O2", date(2023, 6, 1), "C1", "P1", "Alice", "Tech", "Phones", 200.0),
        ("O3", date(2024, 1, 1), "C1", "P1", "Alice", "Tech", "Phones", 50.0),
    ]
    cols = ["order_id", "order_date", "customer_id", "product_id",
            "customer_name", "category", "sub_category", "profit"]
    df = spark.createDataFrame(data, cols)
    result = (
        df.withColumn("year", F.year(F.col("order_date")))
        .groupBy("year", "category", "sub_category", "customer_name")
        .agg(F.sum("profit").alias("total_profit"))
    )
    rows = {r["year"]: r["total_profit"] for r in result.collect()}
    assert rows[2023] == 300.0, f"2023 profit should be 300, got {rows[2023]}"
    assert rows[2024] == 50.0, f"2024 profit should be 50, got {rows[2024]}"

In [0]:
# =======================================================================
# Collect test functions from notebook, generate a test module, run pytest
# =======================================================================

test_dir = "/tmp/ecom_pipeline_tests"
os.makedirs(test_dir, exist_ok=True)

# ---- 1. Read utility source from the utils notebook on disk ----
utils_nb_path = "/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_analysis/resources/utils"
utils_code = ""

for ext in [".ipynb", ""]:
    try:
        with open(utils_nb_path + ext) as f:
            content = f.read()
        if content.strip().startswith("{"):  # Jupyter notebook format
            nb = json.loads(content)
            utils_code = "\n\n".join(
                "".join(cell["source"])
                for cell in nb["cells"]
                if cell["cell_type"] == "code"
                and not "".join(cell.get("source", [])).strip().startswith("%")
            )
            break
    except (FileNotFoundError, json.JSONDecodeError, KeyError):
        continue

# Fallback: extract source via inspect from %run-loaded globals
if not utils_code:
    _util_names = [
        "DataQualityError", "json_reader", "csv_reader", "excel_reader",
        "apply_schema", "delta_writer", "_validate_config",
        "ingest_data", "run_ingestion",
        "validate_not_null", "validate_no_duplicates",
        "validate_schema", "validate_row_count", "run_quality_checks",
        "clean_dataframe",
    ]
    parts = []
    for name in _util_names:
        obj = globals().get(name)
        if obj:
            try:
                parts.append(inspect.getsource(obj))
            except OSError:
                pass
    utils_code = "\n\n".join(parts)
    utils_code += '\n_READERS = {"csv": csv_reader, "json": json_reader, "excel": excel_reader}\n'
    utils_code += '_REQUIRED_CONFIG_KEYS = {"file_path", "file_format", "target_table"}\n'

# ---- 2. Collect test function sources from notebook globals ----
test_names = sorted(n for n in globals() if n.startswith("test_") and callable(globals()[n]))
test_sources = []
for name in test_names:
    try:
        test_sources.append(inspect.getsource(globals()[name]))
    except OSError:
        print(f"\u26a0\ufe0f  Could not extract source for {name}")

# ---- 3. Assemble self-contained test module ----
header = textwrap.dedent("""\
    import pytest
    import logging
    from datetime import date
    from decimal import Decimal
    from pyspark.sql import SparkSession, DataFrame
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
    from pyspark.sql import Window as W
    from pyspark.sql.types import (
        StructType, StructField, StringType, IntegerType,
        DoubleType, DateType, DecimalType, LongType,
    )

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    logger = logging.getLogger("ecommerce_pipeline")

    spark = SparkSession.builder.getOrCreate()

    TEST_CATALOG = "az_ci_de_dbs"
    TEST_SCHEMA  = "ecom_dev"
    def _tbl(name):
        return f"{TEST_CATALOG}.{TEST_SCHEMA}._test_{name}"
""")

test_file = (
    header
    + "\n# " + "=" * 70 + "\n# UTILITY FUNCTIONS\n# " + "=" * 70 + "\n\n"
    + utils_code
    + "\n\n# " + "=" * 70 + "\n# TEST FUNCTIONS\n# " + "=" * 70 + "\n\n"
    + "\n\n".join(test_sources)
)

test_file_path = os.path.join(test_dir, "test_pipeline.py")
with open(test_file_path, "w") as f:
    f.write(test_file)

print(f"\u2705 Test file generated: {test_file_path}")
print(f"   Collected {len(test_names)} test functions")
print(f"   Utils source: {len(utils_code):,} chars")
print("=" * 70)

# ---- 4. Run pytest ----
exit_code = pytest.main([
    test_dir,
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
])

# ---- 5. Summary ----
print("\n" + "=" * 70)
status = "ALL TESTS PASSED \u2705" if exit_code == pytest.ExitCode.OK else "SOME TESTS FAILED \u274c"
print(f"RESULT: {status}  (pytest exit code: {exit_code})")
print("=" * 70)

In [0]:
# -----------------------------------------------------------------------
# Cleanup: Drop all _test_ tables created during testing
# -----------------------------------------------------------------------
test_tables = [
    _tbl("writer_basic"),
    _tbl("writer_overwrite"),
    _tbl("ingest_csv"),
]
for t in test_tables:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        logger.info("Dropped test table: %s", t)
    except Exception as e:
        logger.warning("Could not drop %s: %s", t, e)

print("Test cleanup complete.")